# SAR → structure → optical color: MRSHaze training

This notebook **clones your GitHub repository** and runs all MRSHaze training in Kaggle. First push the contents of this project to your repository; update `REPO_URL` below if needed.

In Kaggle Settings: enable a GPU and Internet, and attach your MRSHaze dataset. No dataset copying or local training is needed. The default URL assumes `sumire25/grayToColor`; it is configurable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Update this if the new repository has a different owner/name.
REPO_URL = "https://github.com/sumire25/SARguidedRSID.git"
REPO_REF = None  # Optional branch/tag. Leave None for the repository default.
REPO_DIR = Path("/kaggle/working/SARguidedRSID")
DATA_ROOT = Path("/kaggle/input/datasets/markosumire/mrshaze/MRSHaze")

if not (REPO_DIR / ".git").is_dir():
    command = ["git", "clone", "--depth", "1"]
    if REPO_REF:
        command += ["--branch", REPO_REF]
    subprocess.run(command + [REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Source revision:", revision)
assert DATA_ROOT.is_dir(), f"Mount MRSHaze and update DATA_ROOT: {DATA_ROOT}"

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU in Kaggle Settings before full training/testing."
print("GPU:", torch.cuda.get_device_name(0))


## Configure or resume

Default: 10 epochs of grayscale recovery, 10 of color warm-up, then 80 of joint training. Validation is a fixed 10% holdout of **train**; the 2,000 test images are never used here. Set `MODE="e2e"` to train jointly from epoch 1, or `"modular"` for the detached-stage control.

For a later Kaggle session, attach the previous notebook output and set `RESUME` to its `last.pt`. Resume reloads the saved config and full state; it preserves the original phase schedule. `STOP_AFTER_EPOCH` is an absolute completed-epoch count, so you can split the run across sessions without redefining its schedule.


In [ ]:
import yaml
from graytocolor.runtime import load_checkpoint

RUN_DIR = Path("/kaggle/working/runs/sar_gray_color")
RESUME = None  # Example: "/kaggle/input/notebooks/markosumire/NOTEBOOK/runs/sar_gray_color/last.pt"
INIT_WEIGHTS = None  # Optional weights-only initialization for a NEW schedule.
STOP_AFTER_EPOCH = None  # Example: 20. Next session resumes with 20 epochs already complete.

MODE = "hybrid"
BATCH_SIZE = 4
CROP_SIZE = 256
ACCUMULATION_STEPS = 1

if RESUME:
    config = load_checkpoint(RESUME)["config"]
else:
    config = yaml.safe_load((REPO_DIR / "mrshaze.yaml").read_text())
    config["mode"] = MODE
    config["data"]["crop_size"] = CROP_SIZE
    config["train"].update(batch_size=BATCH_SIZE, accumulation_steps=ACCUMULATION_STEPS)
    # Architecture/loss/phase ablations can be changed here before a NEW run.
    # config["model"]["use_sar"] = False
    # config["model"]["use_structure_features"] = False

config.update(run_dir=str(RUN_DIR), device="cuda")
config["data"]["root"] = str(DATA_ROOT)
RUN_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RUN_DIR / "kaggle_config.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
(RUN_DIR / "source_revision.txt").write_text(revision + "\n")
print(CONFIG_PATH.read_text())


In [ ]:
command = [sys.executable, "-u", "train.py", "--config", str(CONFIG_PATH)]
if RESUME:
    command += ["--resume", str(RESUME)]
if INIT_WEIGHTS:
    command += ["--init-weights", str(INIT_WEIGHTS)]
if STOP_AFTER_EPOCH is not None:
    command += ["--stop-after-epoch", str(STOP_AFTER_EPOCH)]
subprocess.run(command, cwd=REPO_DIR, check=True)


## Inspect training artifacts

`best.pt` is selected only during the final configured phase. Warm-up checkpoints are saved separately as `best_gray.pt` and `best_color.pt`. For continuation use `last.pt`, which includes optimizer, AMP scaler, RNG state, exact phase position, and split IDs.

Use **Save Version → Save & Run All** to persist notebook output. Attach that output to `test_kaggle.ipynb` for held-out test evaluation. A session stopped during a warm-up may not have `best.pt` yet.


In [ ]:
import json
import matplotlib.pyplot as plt

records = [json.loads(line) for line in (RUN_DIR / "metrics.jsonl").read_text().splitlines()]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([r["epoch"] for r in records], [r["train"]["total"] for r in records])
axes[0].set(xlabel="Epoch", ylabel="Training loss")
for key, label in [("gray_psnr", "Recovered luminance"), ("psnr", "Dehazed RGB")]:
    selected = [r for r in records if key in r["val"]]
    axes[1].plot([r["epoch"] for r in selected], [r["val"][key] for r in selected], label=label)
axes[1].set(xlabel="Epoch", ylabel="Validation PSNR (dB)")
axes[1].legend()
plt.show()
print("Saved artifacts:")
for path in sorted(RUN_DIR.iterdir()):
    print(path.name)
